### Import packages!

In [1]:
!hostname

pe2cl2-002.c.nygenome.org


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from datetime import datetime
import scanpy as sc
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score
from matplotlib.font_manager import FontProperties
import anndata as ad
from scipy.stats import ranksums
import matplotlib.cm as cm
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import umap
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
import statsmodels.api as sm
from statsmodels.stats.multitest import fdrcorrection
import sys
import gffutils
import os

# Add the directory containing IsovizPy.py to the Python path for visualization of ATSEs and isoforms 
sys.path.append('/gpfs/commons/home/kisaev/Leaflet-private/src/visualization')
import IsovizPy as ja

sys.path.append('/gpfs/commons/home/kisaev/Leaflet-analysis')
# Import functions from utilities.py 
from LeafletFA_utilities import *
import importlib

# Increase the font scale for seaborn
sns.set(font_scale=1.5)  # Adjust this value to increase or decrease font size
sns.set_style("white")  # Set the background to white

In [3]:
# Load data and create the database (note this may take 1-2 minutes)
gtf_file = "/gpfs/commons/datasets/controlled/BRAIN_NeMO/human-reference/gencode/gencode.v45.primary_assembly.annotation.gtf"
#db = ja.create_db(gtf_file, "gencode.v45") #do this only once! 

# Path to the database file created previously
db_path = "gencode.v45"

# Load the database
db = gffutils.FeatureDB(db_path, keep_order=True)

### Load all datasets!

In [4]:
# Specify whether GTF file was used during ATSE mapping (whether splice junctions are annotated to genes...)
gtf_used = True

# Today's date 
today = datetime.today().strftime('%Y%m%d')
# Make new directory within current working directory to store all the figures and resutls that will be generated in this notebook
results_dir = f"results_{today}"
os.makedirs(results_dir, exist_ok=True)
os.chdir(results_dir)

# Specify path that contains results from model training 
model_training_results_path = "/gpfs/commons/projects/knowles_singlecell_splicing/PRJEB14362/LeafletFA/analysis_20241203_135345_K_50_UsingWaypoints_Prior_GlobalPrior_NumEpochs_500_LearnedConc_Inits_1_CellType_day_Random_47072"
K = 50 # Number of learned factors 

WD="/gpfs/commons/projects/knowles_singlecell_splicing/PRJEB14362/LeafletFA/ATSEs"
# Note: this ATSE file was also generated through the script mentioned above
atses="iPSC_human_cells_with_annotations_50_500000_500_20241126_single_cell.gz"
ATSE_file=f"{WD}/{atses}"

In [5]:
# Splicing anndata object 
anndata = "ATSE_Anndata_noGTF_Object_20241127_151430.h5ad"
input_file = f"{WD}/{anndata}"

adata_full = ad.read_h5ad(input_file)
splice_adata = adata_full.copy()
splice_adata.obs.reset_index(drop=True, inplace=True)
splice_adata.obs["cell_id_index"] = splice_adata.obs.index 

# Load the expression data (.h5ad file)
exp_file = "/gpfs/commons/projects/knowles_singlecell_splicing/PRJEB14362/zenodo/gene_expression_2024-12-04.h5ad"
adata = sc.read_h5ad(exp_file)

# Mapped ATSE coordinates 
atses = pd.read_csv(ATSE_file, sep="}")

/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [6]:
# Load the latent variables (pickle file)
latent_vars_file = f"{model_training_results_path}/latent_vars.pkl"
latent_vars = pd.read_pickle(latent_vars_file)

# Specify prined latent variables file using 
pruned_latent_vars_file = f"{model_training_results_path}/pruned_latent_vars.pkl"
pruned_vars = pd.read_pickle(pruned_latent_vars_file)

# Load the all_vs_all results (CSV file)
albf_scores_file = f"{model_training_results_path}/all_vs_all_results.csv"
albf_scores = pd.read_csv(albf_scores_file)
albf_scores["junction_id_index"] = albf_scores["Junction_Index"]

# If GTF file was used, specify columns that include "gene_id" otherwise no "gene_id"
if gtf_used:
    juncs_factors_albf = splice_adata.var[["junction_id", "gene_id", "Cluster", "junction_id_index"]].merge(albf_scores,on="junction_id_index")
else:
    juncs_factors_albf = splice_adata.var[["junction_id", "Cluster", "junction_id_index"]].merge(albf_scores,on="junction_id_index")

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

### Assemble all aging related genes

In [ ]:
# Aging genes from eLife paper
age_elife = "/gpfs/commons/groups/knowles_lab/Karin/elife-62293-supp3-v2.xlsx"
age_elife = pd.read_excel(age_elife, header=0, index_col=None)

# Genes for which the fraction of cells expressing significantly increases with age
supptab4 = "/gpfs/commons/groups/knowles_lab/Karin/41586_2020_2496_MOESM6_ESM.xlsx"
supptab4 = pd.read_excel(supptab4, header=0, index_col=None, sheet_name="all_genes")

# Microglia AD signature vs healthy, not sure how to interpret the 500 genes here...  
supptab10 = "/gpfs/commons/groups/knowles_lab/Karin/41586_2020_2496_MOESM12_ESM.xlsx"
supptab10 = pd.read_excel(supptab10, header=1, index_col=None, sheet_name="Amit_expression_mic3tomic1")

# Global aging genes from TMS paper
global_aging_genes = "/gpfs/commons/groups/knowles_lab/Karin/27857814" # from https://figshare.com/articles/dataset/tms_gene_data_rv1/12827615?file=27857814
global_aging_genes = pd.read_csv(global_aging_genes, sep='\t')

# TMS genes 
tms_paper = global_aging_genes["global_aging_genes"].unique()
age_elife_global = age_elife[age_elife["global_aging"]].gene.unique()

# Combine both lists into one, then use set() to remove duplicates
combined_genes = list(set(tms_paper.tolist() + age_elife_global.tolist()))

# Print the number of unique genes
print(f"Total unique genes: {len(combined_genes)}")

### Load list of RBPs and senescence markers!

In [ ]:
# RBP file
excel_file = "/gpfs/commons/groups/knowles_lab/Karin/VanNostrand_2020_supptable1_41586_2020_2077_MOESM3_ESM.xlsx"

# Load the Excel file
rbps = pd.read_excel(excel_file, header=1, index_col=None)
# Rename the first two columns to be gene_name and gene_id 
rbps.rename(columns={rbps.columns[0]: 'gene_name', rbps.columns[1]: 'gene_id'}, inplace=True)

# Convert the gene_name column to capitalize only the first letter and lowercase the rest
rbps['mouse_gene_name'] = rbps['gene_name'].str.capitalize()
print(rbps[['gene_name', 'mouse_gene_name', 'gene_id']].head())

### Load latent splicing factors!

In [ ]:
assign_post = pruned_vars["pruned_assign_post"]
psi_learned = pruned_vars["pruned_psi_learned"]
pi = pruned_vars["pruned_pi"]

# let's get a new junc_ratio by multiplying psi_assigned and learned PSI! then use that to plot junnction usagea cross cells from different age groups.
new_junc_ratio = assign_post @ psi_learned
splice_adata.layers["new_junc_ratio"] = new_junc_ratio

psi_learned = psi_learned
psi_learned_df = pd.DataFrame(psi_learned.T, columns=[f'factor_{i}' for i in range(1, K + 1)])
psi_learned_df["junction_id_index"] = psi_learned_df.index
psi_learned_df = psi_learned_df.merge(splice_adata.var, on="junction_id_index")
splice_adata.var = psi_learned_df

### Figure out which latent spaces are associated with aging

In [ ]:
# Step 1: Convert assign_post to a DataFrame and name the columns as 'factor_0', 'factor_1', ..., 'factor_49'
factor_names = [f'factor_{i}' for i in range(1, K + 1)]
assign_post_df = pd.DataFrame(assign_post, columns=factor_names)
assign_post_df.index = splice_adata.obs.index
# Step 2: Attach the factor names DataFrame (assign_post_df) to adata.obs
splice_adata.obs = pd.concat([splice_adata.obs, assign_post_df], axis=1)

### Run logistic regression to predict age as a categorical variable! 

In [ ]:
# keep only common cells between splice_adata and adata 
adata = adata[adata.obs["cell_name"].isin(splice_adata.obs["cell_name"])]
splice_adata = splice_adata[splice_adata.obs["cell_name"].isin(adata.obs["cell_name"])]

In [ ]:
# Convert 'order_cells' to a plain list or pandas Index
order_cells = splice_adata.obs["cell_name"].tolist()

# Subset and reorder adata to match 'order_cells'
adata = adata[order_cells]

splice_adata.obs["pseudo"] = adata.obs["pseudo"].values
splice_adata.obs["dev_stage"] = adata.obs["dev_stage"].values

In [ ]:
splice_adata.obs

In [ ]:
# Assuming splice_adata is your AnnData object and the latent factors are stored in `splice_adata.obs`
feature = "experiment"
accuracy, coefficients_day = logistic_regression_feature_prediction_simple(splice_adata, feature, K)

# Check the accuracy and coefficients
print(f"Accuracy for {feature}: {accuracy}")

# For each cell type, we will take the top 10 factors by absolute coefficient value.
top_factors_by_day = coefficients_day.apply(lambda x: x.abs().nlargest(10).index.tolist(), axis=1)
print(top_factors_by_day)

In [ ]:
# Assuming splice_adata is your AnnData object and the latent factors are stored in `splice_adata.obs`
feature = "donor_short_id"
accuracy, coefficients_day = logistic_regression_feature_prediction_simple(splice_adata, feature, K)

# Check the accuracy and coefficients
print(f"Accuracy for {feature}: {accuracy}")

# For each cell type, we will take the top 10 factors by absolute coefficient value.
top_factors_by_day = coefficients_day.apply(lambda x: x.abs().nlargest(10).index.tolist(), axis=1)
print(top_factors_by_day)

In [ ]:
# Assuming splice_adata is your AnnData object and the latent factors are stored in `splice_adata.obs`
feature = "day"
accuracy, coefficients_day = logistic_regression_feature_prediction_simple(splice_adata, feature, K)

# Check the accuracy and coefficients
print(f"Accuracy for {feature}: {accuracy}")

# For each cell type, we will take the top 10 factors by absolute coefficient value.
top_factors_by_day = coefficients_day.apply(lambda x: x.abs().nlargest(10).index.tolist(), axis=1)
print(top_factors_by_day)

plot_clustermap(coefficients_day, highlighted_factors=['factor_7', 'factor_16', 'factor_30', 'factor_1'], figsize=(10, 10))

In [ ]:
plot_top_factor_distributions(splice_adata, top_factors=['factor_13', 'factor_16', 'factor_19', 'factor_41'], age_column="day", plot_type="boxenplot")

In [ ]:
# Assuming splice_adata is your AnnData object and the latent factors are stored in `splice_adata.obs`
feature = "dev_stage"
accuracy, coefficients_dev = logistic_regression_feature_prediction_simple(splice_adata, feature, K)
print(coefficients_dev)

# Check the accuracy and coefficients
print(f"Accuracy for {feature}: {accuracy}")

# For each cell type, we will take the top 10 factors by absolute coefficient value.
top_factors_by_dev = coefficients_dev.apply(lambda x: x.abs().nlargest(10).index.tolist(), axis=1)
print(top_factors_by_dev)

# reorder coefficients_dev to match the order of dev_stage
coefficients_dev = coefficients_dev.reindex(["iPSC", "mesendo", "int_unassigned", "defendo"])

plot_clustermap(coefficients_dev, highlighted_factors=['factor_29', 'factor_7', 'factor_16', 'factor_1'], figsize=(10, 10))

### Get UMAP based on assign_post embeddings in splice_data.obs columns with "factor" in them 

In [ ]:
# Step 1: Extract the assign_post embeddings from splice_data.obs
# Get all columns with 'factor' in their names
factor_columns = [col for col in splice_adata.obs.columns if 'factor' in col]
embedding = splice_adata.obs[factor_columns].values

# Step 2: Perform UMAP on the factor embeddings
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
umap_embedding = reducer.fit_transform(embedding)
splice_adata.obsm['X_umap'] = umap_embedding

shuffled_indices = np.random.permutation(splice_adata.obs.index)
shuffled_adata = splice_adata[shuffled_indices, :]

In [ ]:
# get gene_id and gene_name from atses to merge onto splice_adata.var
splice_adata.var = splice_adata.var.merge(atses[["junction_id", "gene_name"]], on="junction_id")

In [ ]:
if gtf_used:
    genes_ids = atses[["gene_id", "gene_name"]].drop_duplicates()
    aging_genes = global_aging_genes[global_aging_genes["global_aging_genes"].isin(splice_adata.var["gene_name"])].global_aging_genes.values
    genes_ids['aging_genes'] = genes_ids['gene_name'].isin(combined_genes).map({True: 'yes', False: 'no'})
    genes_ids.aging_genes.value_counts()
    juncs_factors_albf = juncs_factors_albf.merge(genes_ids)

### Look at ALBF scores of junction within a specific factor

In [ ]:
# Step 1: Select factor of interest 
factor_int = "Factor 16" 

if gtf_used:

    subset_df = juncs_factors_albf[juncs_factors_albf["Factor"] == factor_int]
    
    # Step 2: Sort first by 'aging_genes' (placing 'yes' first) and then by 'ALBF' in descending order
    sorted_df = subset_df.sort_values(by=["aging_genes", "ALBF"], ascending=[False, False])
    sorted_df = subset_df.sort_values(by="ALBF", ascending=False)
    
    # Step 2: Select the top 50 junctions
    top_50_junctions = sorted_df.head(20)
    
    # Step 3: Create a bar plot to show the top 50 junctions and their ALBF scores
    plt.figure(figsize=(7, 9))
    
    # Define colors based on the aging_genes column ('yes' for aging genes, 'no' for non-aging genes)
    color_palette = {'yes': 'red', 'no': 'blue'}
    
    # Plot the barplot with different colors based on the aging_genes column
    sns.barplot(x="ALBF", y="junction_id", data=top_50_junctions, hue="aging_genes", dodge=False, palette=color_palette)
    
    # Italicize gene names
    italic_font = FontProperties(style='italic')
    
    # Add the gene names as labels for each junction with italic font
    for i in range(top_50_junctions.shape[0]):
        plt.text(top_50_junctions["ALBF"].values[i] + 1, i, 
                 top_50_junctions["gene_name"].values[i], 
                 fontproperties=italic_font, color='black', va="center", fontsize=12)
    
    # Set labels and title
    plt.xlabel("ALBF Score", fontsize=14)
    plt.ylabel("Junction ID", fontsize=14)
    plt.yticks(fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{factor_int}_top_junctions.pdf", format='pdf')
    
    # Show the plot
    plt.show()

### Gets PCs and UMAP on gene expression anndata object

In [ ]:
# Identify highly variable genes
sc.pp.highly_variable_genes(adata, flavor='seurat', n_top_genes=2000)

# Subset to highly variable genes 
adata = adata[:, adata.var['highly_variable']]

# PCA computation
sc.tl.pca(adata, n_comps=50)  

# Compute UMAP
sc.pp.neighbors(adata, n_pcs=30, use_rep='X_pca')  # Construct neighborhood graph
sc.tl.umap(adata)  # Compute UMAP embedding

# Plot UMAP
sc.pl.umap(adata, color=['dev_stage']) 

In [ ]:
# plot gene expression PCA 
sc.pl.pca(adata, color=['dev_stage', 'day'])

In [ ]:
sc.pl.umap(adata, color=['day', 'dev_stage']) 

In [ ]:
# plot splicing UMAP 
sc.pl.umap(splice_adata, color=['day', 'dev_stage']) 

### Rerun logistic regression to predict age groups comparing splicing vs gene expression based signals

In [ ]:
def logistic_regression_feature_prediction(
    splice_adata,
    gene_expression_adata,
    feature: str,
    K: int = 50,
    n_pcs: int = 50,
    covariate_column: str = None,  # Allow any covariate column
    test_size: float = 0.2,
    random_state: int = 42
):
    """
    Train and evaluate multinomial logistic regression models using splicing factors,
    gene expression PCs, a covariate column, and their combination to predict the feature of interest.
    
    Parameters:
    - splice_adata: AnnData object containing splicing data with latent factors in `obs`.
    - gene_expression_adata: AnnData object containing gene expression data.
    - feature: str, name of the feature to predict (e.g., 'age', 'cell_type_grouped').
    - K: int, number of splicing latent factors to use.
    - n_pcs: int, number of gene expression PCs to use.
    - covariate_column: str, name of the covariate column in `splice_adata.obs`.
    - test_size: float, proportion of the data to use for testing.
    - random_state: int, seed for reproducibility.
    
    Returns:
    - accuracies: dict, accuracy scores for each model.
    - aurocs: dict, AUROC scores for each model.
    - auprs: dict, AUPR scores for each model.
    - models: dict, trained LogisticRegression models.
    - label_encoder: LabelEncoder instance used to encode the target variable.
    - coefficients_dfs: dict, DataFrames with logistic regression coefficients for each model.
    """

    # Step 1: Extract splicing latent factors
    latent_factors = [f'factor_{i}' for i in range(1, K + 1)]
    X_splicing = splice_adata.obs[latent_factors]
    
    # Step 2: Compute gene expression PCs
    pcs = gene_expression_adata.obsm['X_pca'][:, :n_pcs]
    pc_columns = [f'PC_{i+1}' for i in range(n_pcs)]
    X_gene_expression = pd.DataFrame(pcs, index=gene_expression_adata.obs_names, columns=pc_columns)
    
    # Step 3: Handle covariate column
    if covariate_column:
        covariate_encoder = OneHotEncoder()
        covariate_data = covariate_encoder.fit_transform(splice_adata.obs[[covariate_column]])
        covariate_columns = [
            f'{covariate_column}_{cat}' for cat in covariate_encoder.categories_[0]
        ]
        X_covariate = pd.DataFrame(
            covariate_data.toarray(), index=splice_adata.obs_names, columns=covariate_columns
        )
        # Add covariate to splicing and gene expression datasets
        X_splicing = pd.concat([X_splicing, X_covariate], axis=1)
        X_gene_expression = pd.concat([X_gene_expression, X_covariate], axis=1)
        # Scale the X_splicing and X_gene_expression after adding the covariate
        scaler = StandardScaler()
        X_splicing = pd.DataFrame(scaler.fit_transform(X_splicing), index=X_splicing.index, columns=X_splicing.columns)
        X_gene_expression = pd.DataFrame(scaler.fit_transform(X_gene_expression), index=X_gene_expression.index, columns=X_gene_expression.columns)
    
    else:
        print("No covariate column provided. Please provide a valid covariate column.")
        

    # Step 4: Prepare combined dataset and scale it
    X_combined = pd.concat([X_splicing, X_gene_expression], axis=1)
    scaler = StandardScaler()
    X_combined_scaled = pd.DataFrame(scaler.fit_transform(X_combined), index=X_combined.index, columns=X_combined.columns)

    # Step 5: Encode the target variable
    label_encoder = LabelEncoder()
    y = splice_adata.obs[feature].astype(str)  # Ensure the feature is a string type
    y_encoded = label_encoder.fit_transform(y)
    
    # Step 6: Split the data
    X_train_s, X_test_s, y_train, y_test = train_test_split(
        X_splicing, y_encoded, test_size=test_size, random_state=random_state
    )
    X_train_g, X_test_g, _, _ = train_test_split(
        X_gene_expression, y_encoded, test_size=test_size, random_state=random_state
    )
    X_train_c, X_test_c, _, _ = train_test_split(
        X_combined_scaled, y_encoded, test_size=test_size, random_state=random_state
    )
    X_train_cov, X_test_cov, _, _ = train_test_split(
        X_covariate, y_encoded, test_size=test_size, random_state=random_state
    )

    # Step 7: Train logistic regression models
    logreg_params = {
        'multi_class': 'multinomial',
        'solver': 'lbfgs',
        'max_iter': 1000,
        'random_state': random_state
    }

    model_splicing = LogisticRegression(**logreg_params)
    model_gene_expression = LogisticRegression(**logreg_params)
    model_combined = LogisticRegression(**logreg_params)
    model_cov = LogisticRegression(**logreg_params)
    
    model_splicing.fit(X_train_s, y_train)
    model_gene_expression.fit(X_train_g, y_train)
    model_combined.fit(X_train_c, y_train)
    model_cov.fit(X_train_cov, y_train)
    
    # Step 8: Evaluate the models
    y_pred_s = model_splicing.predict(X_test_s)
    y_pred_g = model_gene_expression.predict(X_test_g)
    y_pred_c = model_combined.predict(X_test_c)
    y_pred_cov = model_cov.predict(X_test_cov)

    accuracy_s = accuracy_score(y_test, y_pred_s)
    accuracy_g = accuracy_score(y_test, y_pred_g)
    accuracy_c = accuracy_score(y_test, y_pred_c)
    accuracy_cov = accuracy_score(y_test, y_pred_cov)
    
    y_prob_s = model_splicing.predict_proba(X_test_s)
    y_prob_g = model_gene_expression.predict_proba(X_test_g)
    y_prob_c = model_combined.predict_proba(X_test_c)
    y_prob_cov = model_cov.predict_proba(X_test_cov)

    try:
        auroc_s = roc_auc_score(y_test, y_prob_s, multi_class='ovr')
        auroc_g = roc_auc_score(y_test, y_prob_g, multi_class='ovr')
        auroc_c = roc_auc_score(y_test, y_prob_c, multi_class='ovr')
        auroc_cov = roc_auc_score(y_test, y_prob_cov, multi_class='ovr')
        
        aupr_s = average_precision_score(y_test, y_prob_s, average='macro')
        aupr_g = average_precision_score(y_test, y_prob_g, average='macro')
        aupr_c = average_precision_score(y_test, y_prob_c, average='macro')
        aupr_cov = average_precision_score(y_test, y_prob_cov, average='macro')
    except ValueError as e:
        print(f"Error calculating AUROC/AUPR: {e}")
        auroc_s = auroc_g = auroc_c = None
        aupr_s = aupr_g = aupr_c = None

    # Print results
    print(f"Splicing Accuracy: {accuracy_s:.4f}, AUROC: {auroc_s}, AUPR: {aupr_s}")
    print(f"Gene Expression Accuracy: {accuracy_g:.4f}, AUROC: {auroc_g}, AUPR: {aupr_g}")
    print(f"Combined Features Accuracy: {accuracy_c:.4f}, AUROC: {auroc_c}, AUPR: {aupr_c}")
    print(f"Covariate Accuracy: {accuracy_cov:.4f}, AUROC: {auroc_cov}, AUPR: {aupr_cov}")

    # Retrain models for coefficients
    model_splicing.fit(X_splicing, y_encoded)
    model_gene_expression.fit(X_gene_expression, y_encoded)
    model_combined.fit(X_combined_scaled, y_encoded)
    model_cov.fit(X_covariate, y_encoded)
    
    coef_splicing = pd.DataFrame(
        model_splicing.coef_, columns=list(X_splicing.columns), index=label_encoder.classes_
    )
    coef_gene_expression = pd.DataFrame(
        model_gene_expression.coef_, columns=list(X_gene_expression.columns), index=label_encoder.classes_
    )
    coef_combined = pd.DataFrame(
        model_combined.coef_, columns=list(X_combined.columns), index=label_encoder.classes_
    )
    coef_cov = pd.DataFrame(
        model_cov.coef_, columns=list(X_covariate.columns), index=label_encoder.classes_
    )
    
    # Collect results
    accuracies = {
        'splicing': accuracy_s,
        'gene_expression': accuracy_g,
        'combined': accuracy_c,
        'covariate': accuracy_cov
    }
    
    aurocs = {
        'splicing': auroc_s,
        'gene_expression': auroc_g,
        'combined': auroc_c,
        'covariate': auroc_cov
    }
    
    auprs = {
        'splicing': aupr_s,
        'gene_expression': aupr_g,
        'combined': aupr_c,
        'covariate': aupr_cov
    }
    
    models = {
        'splicing': model_splicing,
        'gene_expression': model_gene_expression,
        'combined': model_combined,
        'covariate': model_cov
    }
    
    coefficients_dfs = {
        'splicing': coef_splicing,
        'gene_expression': coef_gene_expression,
        'combined': coef_combined,
        'covariate': coef_cov
    }
    
    return accuracies, aurocs, auprs, models, label_encoder, coefficients_dfs


In [ ]:
splice_adata.obs.set_index("cell_name", inplace=True)

In [ ]:
# get linear regression estimate of how well factors predict pseudo column 
feature = "pseudo"
latent_factors = [f'factor_{i}' for i in range(1, K + 1)]
X = splice_adata.obs[latent_factors]
y = splice_adata.obs[feature]
X = sm.add_constant(X)

# Split the data into training and testing sets 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train multivariate linear regression model
model = sm.OLS(y_train, X_train).fit()

# Evaluate the model on test set 
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print(f"Mean Squared Error: {mse}")
print(f"Root Mean Squared Error: {rmse}")

# get coefficients from the full model trained aon full dataset
linreg_full = sm.OLS(y, X).fit()
# Get the linear regression coefficients, confidence intervals, and p-values
coefficients = linreg_full.params
conf_intervals = linreg_full.conf_int()
std_errors = linreg_full.bse

# Create a DataFrame with coefficients and confidence intervals
coefficients_df = pd.DataFrame({
        'Factor': ['Intercept'] + latent_factors,
        'Coefficient': coefficients,
        'Lower CI': conf_intervals[0],
        'Upper CI': conf_intervals[1],
        'Standard Error': std_errors
    })

In [ ]:
# sort by absolute coefficient value
coefficients_df = coefficients_df.reindex(coefficients_df['Coefficient'].abs().sort_values(ascending=False).index)
coefficients_df.head()

In [ ]:
# plot correlation between Factor_36 cell activity and pseudo column
plt.figure(figsize=(8, 6))
sns.scatterplot(x="factor_34", y="factor_36", data=splice_adata.obs)
plt.xlabel("factor_34")
plt.ylabel("Pseudo")

In [ ]:
# Assume splice_adata and gene_expression_adata are already loaded and preprocessed
feature_to_predict = 'dev_stage' 

accuracies, aurocs, auprs, models, label_encoder, coefficients_dfs = logistic_regression_feature_prediction(
    splice_adata,
    adata,
    feature=feature_to_predict,
    covariate_column = "experiment",
    K=K,
    n_pcs=50,
    test_size=0.3,
    random_state=4100
)

print(accuracies)

In [ ]:
coefficients_dfs.keys()

In [ ]:
coef_splicing = coefficients_dfs['splicing']
coef_gene_expression = coefficients_dfs['gene_expression']
coef_combined = coefficients_dfs['combined']
coef_cov_only = coefficients_dfs['covariate']

In [ ]:
# Assume splice_adata and gene_expression_adata are already loaded and preprocessed
feature_to_predict = 'day' 

accuracies, aurocs, auprs, models, label_encoder, coefficients_days = logistic_regression_feature_prediction(
    splice_adata,
    adata,
    feature=feature_to_predict,
    covariate_column = "experiment",
    K=K,
    n_pcs=50,
    test_size=0.3,
    random_state=4100
)

print(accuracies)

In [ ]:
coef_splicing

In [ ]:
coef_gene_expression

### Calcualte Entropy and Perplexity across Cells

In [ ]:
# subset assign_post to just the cell_index still present in splice_adata.obs 
assign_post = assign_post[splice_adata.obs["cell_id_index"], :]

In [ ]:
# Calculate entropy for each cell (each row sums to 1)
entropy = -np.sum(assign_post * np.log(assign_post + 1e-12), axis=1)  # Small epsilon added for numerical stability

# Calculate perplexity for each cell
perplexity = np.exp(entropy)

# Combine results for better readability
results = np.column_stack((entropy, perplexity))

# Display the first few results
print("Entropy and Perplexity for each cell (first 10 cells):")
print(results[:10])

splice_adata.obs["perplexity"] = perplexity
splice_adata.obs["entropy"] = entropy

# Assuming `perplexity` is already added to `splice_adata.obs` as a column
sns.histplot(data=splice_adata.obs, x="perplexity", hue="dev_stage", bins=30, kde=True, element="step")
plt.xlabel("Perplexity (Number of Factors Active)", fontsize=14)
plt.ylabel("Number of Cells", fontsize=14)
plt.title("Distribution of Perplexity Across Cells by Age", fontsize=14)
plt.show()

In [ ]:
# Assuming `perplexity` is already added to `splice_adata.obs` as a column
sns.histplot(data=splice_adata.obs, x="entropy", hue="day", bins=100, kde=True, element="step")
plt.xlabel("Cell Entropy", fontsize=14)
plt.ylabel("Number of Cells", fontsize=14)
plt.title("Distribution of Entropy Across Cells by Age", fontsize=14)
plt.show()

In [ ]:
splice_adata

### Calcualte Entropy and Perplexity across Junctions!

In [ ]:
# remove index names from splice_adata.var 
splice_adata.var = splice_adata.var.reset_index(drop=True)

In [ ]:
# Calculate entropy for each junction (columns represent factors for each junction)
entropy = -np.sum(psi_learned.T * np.log(psi_learned.T + 1e-12), axis=1)

# Calculate perplexity for each junction
perplexity = np.exp(entropy)

# Combine results for better readability
results = np.column_stack((entropy, perplexity))

# Make into dataframe with columns junction_entropt and junction_perplexity
junc_results = pd.DataFrame(results, columns=["junction_entropy", "junction_perplexity"])
junc_results["junction_id"] = splice_adata.var["junction_id"]
junc_results.sort_values(by="junction_perplexity", ascending=False, inplace=True)
junc_results

In [ ]:
splice_adata.var = splice_adata.var.merge(junc_results, on="junction_id")

In [ ]:
# drop these columns from splice_adat.var junction_entropy_x	junction_perplexity_x	junction_entropy_y	junction_perplexity_y
# splice_adata.var = splice_adata.var.drop(columns=["junction_entropy_x", "junction_perplexity_x", "junction_entropy_y", "junction_perplexity_y"])

In [ ]:
# drop Junction_Index from albf_scores 
albf_scores = albf_scores.drop(columns=["Junction_Index"])

In [ ]:
albf_scores

In [ ]:
splice_adata.var.sort_values(by="junction_id_index", ascending=False, inplace=True)
splice_adata.var.sort_values(by="junction_perplexity", ascending=True, inplace=False)

### Calcualte Entropy and Perplexity across Factors to see how many cell types they are active in ?

### Look at factor-factor correlations in terms of cell usage - also do this usage junction PSI

In [ ]:
splice_adata.obs[factor_columns].shape

In [ ]:
factor_corr_spearman_all = splice_adata.obs[factor_columns].corr(method='spearman')

# Create a mask to hide the diagonal
mask = np.eye(len(factor_corr_spearman_all), dtype=bool)

# Set overall font size smaller using Seaborn context
sns.set_context("paper", font_scale=1.2)  # Adjust `font_scale` for smaller/larger font

# Create a clustermap with masked diagonal
g = sns.clustermap(
    factor_corr_spearman_all, annot=False, cmap='coolwarm', center=0, mask=mask,
    yticklabels=1, xticklabels=1, figsize=(6, 6), vmin=-1, vmax=1)

plt.title("Factor Correlations \nAll Cells")
plt.show()

In [ ]:
# let's look at correlation of cells activities among factor_1 and factor_13 

# Plot scatterplot with density visualization
sns.kdeplot(
    data=splice_adata.obs,
    x="factor_36",
    y="factor_34",
    levels=5, thresh=.2,
    hue="day")
plt.show()

### Look at factor-factor correlations using junction PSI values...

In [ ]:
psi_learned_df = pd.DataFrame(psi_learned).T
psi_learned_df.head()

In [ ]:
sns.kdeplot(
    data=psi_learned_df.sample(5000),x=33, y=35)
plt.show()

In [ ]:
factor_corr_spearman_juncs = psi_learned_df.corr(method='spearman')

In [ ]:
# Create a mask to hide the diagonal
mask = np.eye(len(factor_corr_spearman_juncs), dtype=bool)

# Set overall font size smaller using Seaborn context
sns.set_context("paper", font_scale=1.2)  # Adjust `font_scale` for smaller/larger font

# Create a clustermap with masked diagonal
g = sns.clustermap(
    factor_corr_spearman_juncs, annot=False, cmap='coolwarm', center=0, mask=mask,
    yticklabels=1, xticklabels=1, figsize=(6, 6), vmin=-1, vmax=1)

plt.title("Factor Correlations \nAll Junctions")
plt.show()

### Look at junction annotations!

In [ ]:
# junction_id  Cluster  usage_ratio
splice_adata_var = juncs_factors_albf[["junction_id", "Cluster"]].drop_duplicates()
splice_adata_var["usage_ratio"] = 0

# Convert junction_ids
splice_junctions = ja.convert_junction_ids(splice_adata_var)

In [ ]:
splice_junctions[0]

In [ ]:
# Check junction annotations
junction_annotation_results = ja.check_junction_annotation(splice_junctions, db)

# Extract unique transcript IDs from junction_labels
unique_transcripts = list({transcript for label in junction_annotation_results for transcript in label['transcripts']})

In [ ]:
print(junction_annotation_results[1])

In [ ]:
junction_annotation_results_df = pd.DataFrame(junction_annotation_results)

# if label_5_prime is "annotated on 5'" or label_3_prime is "annotated on 3'", then add annotation columns where values would be "one side" if both then "both sides" and if none then "none"
junction_annotation_results_df["annotation"] = "none"
junction_annotation_results_df.loc[(junction_annotation_results_df["label_5_prime"] == "annotated on 5'"), "annotation"] = "one side"
junction_annotation_results_df.loc[(junction_annotation_results_df["label_3_prime"] == "annotated on 3'"), "annotation"] = "one side"
# now check if both are annotated
junction_annotation_results_df.loc[(junction_annotation_results_df["label_5_prime"] == "annotated on 5'") & (junction_annotation_results_df["label_3_prime"] == "annotated on 3'"), "annotation"] = "both sides"
junction_annotation_results_df.annotation.value_counts()

In [ ]:
# save junction_annotation_results_df to file 
junction_annotation_results_df.to_csv("junction_annotation_results_df.csv")

In [ ]:
junction_annotation_results_df

In [ ]:
# add original junction_id to the junction_annotation_results_df
junction_annotation_results_df["junction_id"] = splice_adata_var["junction_id"].values

In [ ]:
# add annotation to splice_adata.var 
splice_adata.var = splice_adata.var.merge(junction_annotation_results_df, on="junction_id")

In [ ]:
plot_top_factor_distributions(splice_adata, top_factors=["factor_17", "factor_4", "factor_2"], age_column="age", plot_type="boxenplot")

In [ ]:
# now let's find top 10% of junctions with highest ALBF scores for factor 2
top_10_percent = int(0.1 * len(splice_adata.var))

# get percent of top junctions that are annotated (both sides) vs (vs one side) vs unannotated for each factor
junc_anno_factors = {}
factor_columns

for factor in factor_columns:
    top_junctions = splice_adata.var.sort_values(by=factor, ascending=False).head(top_10_percent)
    junc_anno_factors[factor] = top_junctions.annotation.value_counts()
    
# convert to dataframe
junc_anno_factors_df = pd.DataFrame(junc_anno_factors).T
junc_anno_factors_df

# get the percentage of annotated junctions for each factor 
junc_anno_factors_df["annotated_percent"] = junc_anno_factors_df["both sides"] / top_10_percent * 100
# get the percentage of unannotated junctions for each factor
junc_anno_factors_df["unannotated_percent"] = junc_anno_factors_df["none"] / top_10_percent * 100
# get the percentage of junctions annotated on one side for each factor
junc_anno_factors_df["one_side_percent"] = junc_anno_factors_df["one side"] / top_10_percent * 100

In [ ]:
plot_top_factor_distributions(splice_adata, top_factors=["factor_26"], age_column="age", plot_type="boxenplot")

In [ ]:
sns.kdeplot(
    data=splice_adata.obs.sample(10000),
    x="factor_2",
    y="factor_12",
    levels=5, thresh=.2,
    hue="age")
plt.show()

In [ ]:
sns.kdeplot(
    data=splice_adata.obs.sample(10000),
    x="factor_2",
    y="factor_26",
    levels=5, thresh=.2,
    hue="age")
plt.show()

In [ ]:
# keep just the columns that have "factor" in them and melt the table 
coef_splicing_df = coef_splicing.filter(like="factor").reset_index().melt(id_vars="index", var_name="factor", value_name="count")
# rename columns to age group, factor and coefficient 
coef_splicing_df.columns = ["age_group", "factor", "coefficient"]
# sort by absolute coefficient value
coef_splicing_df = coef_splicing_df.sort_values(by="coefficient", ascending=False)
coef_splicing_df

In [ ]:
# let's extract junction_id and perxplecity from splice_adata.var and merge with juncs_factors_albf
splice_adata_var = splice_adata.var[["junction_id", "junction_perplexity"]].drop_duplicates()
juncs_factors_albf = juncs_factors_albf.merge(splice_adata_var, on="junction_id")

In [ ]:
juncs_factors_albf.sort_values(by="ALBF", ascending=False)

In [ ]:
# let's find top 5 cell_types and label them and all other cells make grey 
top_5_cell_types = splice_adata.obs["cell_type_grouped"].value_counts().head(20).index.values

# create a new column in splice_adata.obs called "cell_type_grouped_top5" and label the top 5 cell types and all other cell types as "other"
splice_adata.obs["cell_type_grouped_top5"] = splice_adata.obs["cell_type_grouped"].apply(lambda x: x if x in top_5_cell_types else "other")

# make umap plot with cell_type_grouped_top5 as color
sc.pl.umap(splice_adata, color=["cell_type_grouped_top5"])

In [ ]:
sc.pl.umap(splice_adata, color=["age", "factor_2", "factor_26"])

In [ ]:
junction_id_index = 35300  
splice_adata.obs['junction_psi'] = splice_adata.layers['new_junc_ratio'][:, junction_id_index].flatten()

# Step 2: Plot UMAP with junction usage ratio
sc.pl.umap(splice_adata, color=['junction_psi'], color_map='coolwarm', size=15)

In [ ]:
junction_id_index = 26682  
splice_adata.obs['junction_psi'] = splice_adata.layers['new_junc_ratio'][:, junction_id_index].flatten()

# Step 2: Plot UMAP with junction usage ratio
sc.pl.umap(splice_adata, color=['junction_psi'], color_map='coolwarm', size=15)

In [ ]:
# make barplot ordered by annotated_percent in decreasing order 
junc_anno_factors_df.sort_values(by="annotated_percent", ascending=False, inplace=True)
plt.figure(figsize=(10, 6))
sns.barplot(data=junc_anno_factors_df, x=junc_anno_factors_df.index, y="annotated_percent")
plt.xticks(rotation=45)
plt.xlabel("Factor")
plt.ylabel("Percentage of Annotated Junctions")
plt.title("Percentage of Annotated Junctions for Top 10% Junctions by Factor")
plt.show()

### Fgfr2 specific visualization of mutually exclusive exons

In [ ]:
splice_adata_var = splice_adata_var[splice_adata_var["Cluster"] == 26488] # Fgfr2 related cluster
# Let's limit to just Fgfr2 mutually exclusive exons isoforms 120187 and 122054
unique_transcripts = ["ENSMUST00000120187.8", "ENSMUST00000122054.7"] # Fgfr2 mutually exclusive exons

# Fetch transcript exon coordinates and determine plot boundaries
transcript_data = ja.fetch_transcripts_and_annotations(unique_transcripts, db)
region_start, region_end = ja.determine_region_boundaries(splice_junctions)
junc_annots = ja.check_junction_annotation(splice_junctions, db)

In [ ]:
ja.plot_exons_and_junctions(db, transcript_data, splice_junctions, region_start, region_end, base_width=10, trans_height=0.3, show_usage=False, show_junc_lines=True)

In [ ]:
junction_id_index = 42968  #chr7_130196374_130198418_- --> 	 
splice_adata.obs['junction_psi'] = splice_adata.layers['new_junc_ratio'][:, junction_id_index].flatten()

# Step 2: Plot UMAP with junction usage ratio
sc.pl.umap(splice_adata, color=['junction_psi'], color_map='coolwarm', size=15)

In [ ]:
junction_id_index = 42969 # chr7_130196374_130199757_- # Replace with the actual index for the junction of interest
splice_adata.obs['junction_psi'] = splice_adata.layers['new_junc_ratio'][:, junction_id_index].flatten()

# Step 2: Plot UMAP with junction usage ratio
sc.pl.umap(splice_adata, color=['junction_psi'], color_map='coolwarm', size=15)